# Layer 3: Exchange Timestamp Analysis (ME Profiling)


In [ ]:
import os
import numpy as np
import pandas as pd

from mda.loader import load_trades
from mda.timestamps import add_ts_columns, classify_resolution
from mda import EXCHANGES, DATA_DIR
from mda.layer3.quantisation import compute_timestamp_gaps, resolution_report
from mda.layer3.execution_rate import compute_execution_rate, execution_rate_stats
from mda.layer3.microbursts import detect_microbursts, microburst_stats, microburst_heatmap
from mda.layer3.latency import (
    compute_feed_latency,
    latency_stats,
    latency_drift_timeseries,
)
from mda.layer3.sequencing import compute_out_of_order
from mda.plots.charts import (
    save_figure,
    plot_timestamp_gap_hist,
    plot_execution_rate_ts,
    plot_microburst_heatmap,
    plot_latency_distribution,
    plot_latency_drift,
    plot_out_of_order_events,
)


In [ ]:
DATA_DIR = "/data/parquet"
REPORTS_DIR = "/data/reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

df = load_trades(DATA_DIR, exchanges=EXCHANGES, add_ts_cols=True)
print("Shape:", df.shape)
print(df[["exchange", "ts_event", "ts_recv"]].head())


In [ ]:
gaps = compute_timestamp_gaps(df)
res = resolution_report(df)

print("Timestamp resolution report:")
print(res)


In [ ]:
# Plot E4: Timestamp gap histogram
fig = plot_timestamp_gap_hist(gaps)
fig.show()
save_figure(fig, "E4_timestamp_gap_hist", REPORTS_DIR)


In [ ]:
exec_rate = compute_execution_rate(df)
exec_stats = execution_rate_stats(exec_rate)

print("Execution rate statistics:")
print(exec_stats)


In [ ]:
# Plot E1: Execution rate timeseries
fig = plot_execution_rate_ts(exec_rate)
fig.show()
save_figure(fig, "E1_execution_rate_ts", REPORTS_DIR)


In [ ]:
bursts = detect_microbursts(df)
mb_stats = microburst_stats(bursts)
mb_heatmap = microburst_heatmap(bursts)

print("Microburst statistics:")
print(mb_stats)


In [ ]:
# Plot E2 and E3: Microburst heatmaps per exchange
for exch in EXCHANGES:
    bursts_ex = bursts[bursts["exchange"] == exch] if "exchange" in bursts.columns else bursts
    heatmap_ex = mb_heatmap.get(exch, mb_heatmap) if isinstance(mb_heatmap, dict) else mb_heatmap

    fig = plot_microburst_heatmap(heatmap_ex, exchange=exch)
    fig.show()
    save_figure(fig, f"E2_microburst_heatmap_{exch}", REPORTS_DIR)

    fig3 = plot_microburst_heatmap(heatmap_ex, exchange=exch, mode="intensity")
    fig3.show()
    save_figure(fig3, f"E3_microburst_intensity_{exch}", REPORTS_DIR)


In [ ]:
df_lat = compute_feed_latency(df)
lat_stats = latency_stats(df_lat)
drift = latency_drift_timeseries(df_lat)

print("Feed latency statistics (microseconds):")
print(lat_stats)


In [ ]:
# Plot E5: Latency distribution; E6: Latency drift
fig5 = plot_latency_distribution(df_lat)
fig5.show()
save_figure(fig5, "E5_latency_distribution", REPORTS_DIR)

fig6 = plot_latency_drift(drift)
fig6.show()
save_figure(fig6, "E6_latency_drift", REPORTS_DIR)


In [ ]:
oo_summary, oo_events = compute_out_of_order(df)

print("Out-of-order event summary:")
print(oo_summary)


In [ ]:
# Plot E7: Out-of-order events
fig7 = plot_out_of_order_events(oo_events)
fig7.show()
save_figure(fig7, "E7_out_of_order_events", REPORTS_DIR)
